In [ ]:
import os

max_gen_tokens = 500
SEED = 83

NUM_RUNS = 50

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import random
from datasets import load_dataset
import string
from datetime import datetime
import csv
import pickle
import shutil
from tqdm import tqdm
# from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import json
from transformers import LogitsProcessor
from transformers import AutoTokenizer, AutoModelForCausalLM


tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-4b-it",
    dtype=torch.bfloat16,
    device_map="cuda"
)
# processor = AutoProcessor.from_pretrained("google/gemma-3-12b-it")
# model = Gemma3ForConditionalGeneration.from_pretrained(
#     "google/gemma-3-12b-it",
#     dtype=torch.bfloat16,
#     device_map="cuda"
# )
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token

class StoreTargetLogitsProcessor(LogitsProcessor):
    """
    store only the logits of the target result
    """
    def __init__(self, target_index, top_k=15):
        self.target_index = target_index
        self.stored_logits = []
        self.top_k = top_k
        
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        target_scores = scores[self.target_index, :].detach().to('cpu')
        vocab_size = target_scores.shape[-1]
        top_values, top_indices = torch.topk(target_scores, self.top_k)
        values = top_values.detach().to('cpu')
        indices = top_indices.detach().to('cpu')
        sparse_indices = indices.unsqueeze(0)
        sparse_logit = torch.sparse_coo_tensor(
            sparse_indices, 
            values, 
            size=(vocab_size,)
        )
        
        self.stored_logits.append(sparse_logit)
        
        return scores

for BATCH_SIZE in [1,2,4,8,16]:
# for BATCH_SIZE in [8, 16]:
    SAVE_DIR  = f"mix_stability_reports_gemma3_4B_B{BATCH_SIZE}"
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    
    set_seed(SEED)
    
    MMLU_SAMPLES_FILE = "mmlu_10_random_samples.jsonl"
    mmlu_dataset = load_dataset("json", data_files=MMLU_SAMPLES_FILE, split="train")
    
    with open("mmlu_1000_random_samples_filler.jsonl", "r", encoding="utf-8") as f:
        filler_questions = [json.loads(line) for line in f]
    
    def generate_noise_prompt_from_filler():

        sample = random.choice(filler_questions)
        question = sample['question']     
        choices = sample['choices'] 
    
    
        formatted_prompt = f"{question}\n\nChoices:\nA. {choices[0]}\nB. {choices[1]}\nC. {choices[2]}\nD. {choices[3]}\nPlease think step by step and then give the final answer.\nAnswer:"
        # print(formatted_prompt)
        # target_message = [{"role": "user", "content": [{"type": "text", "text": formatted_prompt}]}]
        return formatted_prompt
    
    for idx, example in enumerate(tqdm(mmlu_dataset, desc="Analysing")):
            
        question = example['question']
        choices = example['choices']
        formatted_prompt = f"{question}\n\nChoices:\nA. {choices[0]}\nB. {choices[1]}\nC. {choices[2]}\nD. {choices[3]}\nPlease think step by step and then give the final answer.\nAnswer:"
    
        # formatted_prompt = "Please generate a random integer between 1 and 10. Only provide the number."
        target_message = [{"role": "user", "content": [{"type": "text", "text": formatted_prompt}]}]
        q_dir = os.path.join(SAVE_DIR, f"question_{idx:03d}")
        os.makedirs(q_dir, exist_ok=True)
        # print(target_message)
        for run in tqdm(range(NUM_RUNS)):
            save_path = os.path.join(q_dir, f"run_{run:02d}.pkl")
            if os.path.exists(save_path):
                print('skip')
                continue
            if BATCH_SIZE == 1 and run > 0:
                break
            batch_messages = []
            target_index = random.randint(0, BATCH_SIZE - 1)
            target_logits_processor = StoreTargetLogitsProcessor(target_index=target_index)
            
            for i in range(BATCH_SIZE):
                if i == target_index:
                    batch_messages.append(target_message)
                else:
                    noise_message = [{"role": "user", "content": [{"type": "text", "text": generate_noise_prompt_from_filler()}]}]
                    batch_messages.append(noise_message)
    
            inputs = tokenizer.apply_chat_template(
                batch_messages, add_generation_prompt=True, tokenize=True,
                return_dict=True, return_tensors="pt", padding=True,
            ).to(model.device)
            
            prompt_length = inputs["input_ids"].shape[-1]
    
            common_generate_args = {
                "max_new_tokens": max_gen_tokens, "output_scores": False,
                "return_dict_in_generate": True, "do_sample": False,
                "temperature": None, "top_p":None, "logits_processor": [target_logits_processor],
            }
    
            with torch.no_grad():
                outputs = model.generate(**inputs, **common_generate_args)
            
            run_tokens_cpu = outputs.sequences[target_index][prompt_length:].to('cpu')
            # run_logits_cpu = [step_logits[target_index, :].to('cpu') for step_logits in outputs.scores]
            run_logits_cpu = target_logits_processor.stored_logits
            decoded_output = tokenizer.decode(run_tokens_cpu, skip_special_tokens=True)
            # print(f"\n--- Run {run} output ---\n{decoded_output}\n")
            with open(save_path, "wb") as f:
                pickle.dump({"tokens": run_tokens_cpu, "logits": run_logits_cpu}, f)
            tqdm.write(f"✅ saved: {save_path}")

            del outputs, run_tokens_cpu, run_logits_cpu, inputs, target_logits_processor
            torch.cuda.empty_cache()

In [ ]:
import torch
print(torch.cuda.is_available())
# 如果输出 True，说明环境配置成功！

In [ ]:
import torch
print(torch.__version__)